In [48]:
import numpy as np

solution = np.load('solution_nx1000_u.npy')
print(solution.shape)


(10076, 1000)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from tqdm.auto import tqdm

def cavity_mode(n, x, L):
    return np.sqrt(2.0 / L) * np.sin(n * np.pi * x / L)

def run_fd_no_time_loop_with_heatmap():
    # -----------------------------
    # Parameters
    # -----------------------------
    L = np.pi
    g = -1.0                 # defocusing; try -1.0 for focusing (may need tighter tolerances)
    N = 1024                 # grid points including boundaries
    x = np.linspace(0.0, L, N)
    dx = x[1] - x[0]

    # interior points only (Dirichlet boundaries fixed at 0)
    xi = x[1:-1]
    Ni = xi.size

    # output times (solver advances internally; you just request outputs)
    Tfinal = 40.0
    Nt_out = 1000
    t_eval = np.linspace(0.0, Tfinal, Nt_out)

    # -----------------------------
    # Finite-difference Laplacian on interior (Dirichlet)
    # -----------------------------
    main = (-2.0 / dx**2) * np.ones(Ni)
    off  = ( 1.0 / dx**2) * np.ones(Ni - 1)
    D2 = diags([off, main, off], offsets=[-1, 0, 1], format="csc")

    # Linear operator for RHS: i/2 * D2
    Lin = (1j / 2.0) * D2

    # -----------------------------
    # Initial condition: two cavity modes (quantized)
    # psi(x,0) = a*phi_1 + b*e^{i phi}*phi_2
    # -----------------------------
    n1, n2 = 1, 2
    a, b = 1.0, 0.6
    phase = 0.5 * np.pi

    psi0_full = a * cavity_mode(n1, x, L) + b * np.exp(1j * phase) * cavity_mode(n2, x, L)
    psi0 = psi0_full[1:-1].astype(np.complex128)

    # -----------------------------
    # RHS for solve_ivp (ODE system)
    # psi_t = (i/2) D2 psi  - i g |psi|^2 psi
    # -----------------------------
    def rhs(t, psi):
        return Lin @ psi - 1j * g * (np.abs(psi)**2) * psi

    # -----------------------------
    # RK4 time stepping with internal substeps (outputs on t_eval)
    # Explicit RK methods need dt ~ O(dx^2) for FD Laplacians.
    # -----------------------------
    dt_out = float(t_eval[1] - t_eval[0])

    # Heuristic stable/accurate internal step size (tune if needed)
    dt_int_target = dx**2

    Psi = np.empty((Ni, Nt_out), dtype=np.complex128)
    Psi[:, 0] = psi0

    psi = psi0
    total_internal_steps = 0
    for k in range(Nt_out - 1):
        nsub = int(np.ceil(dt_out / dt_int_target))
        total_internal_steps += nsub

    pbar = tqdm(total=total_internal_steps, desc="RK4", unit="step")
    try:
        for k in range(Nt_out - 1):
            t0 = float(t_eval[k])
            nsub = int(np.ceil(dt_out / dt_int_target))
            dt = dt_out / nsub

            t = t0
            for _ in range(nsub):
                k1 = rhs(t, psi)
                k2 = rhs(t + 0.5 * dt, psi + 0.5 * dt * k1)
                k3 = rhs(t + 0.5 * dt, psi + 0.5 * dt * k2)
                k4 = rhs(t + dt, psi + dt * k3)

                psi = psi + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
                t += dt
                pbar.update(1)

            Psi[:, k + 1] = psi
    finally:
        pbar.close()

    # -----------------------------
    # Modal projection (vectorized)
    # c_n(t) = ∫ phi_n(x) psi(x,t) dx ≈ dx * phi^H psi
    # -----------------------------
    Nmodes = 80  # heatmap height; increase if you want deeper cascades
    modes = np.arange(1, Nmodes + 1)

    Phi = np.stack([cavity_mode(n, xi, L) for n in modes], axis=0)  # (Nmodes, Ni)
    C = dx * (Phi.conj() @ Psi)                                     # (Nmodes, Nt_out)
    En = np.abs(C)**2                                               # (Nmodes, Nt_out)

    # Optional: log-scale for visibility (avoid log(0))
    En_log = np.log10(En + 1e-16)

    # -----------------------------
    # Plot 1: a few mode energies
    # -----------------------------
    plt.figure()
    for i, n in enumerate([1, 2, 3, 4, 5, 6]):
        plt.plot(t_eval, En[n-1], label=f"n={n}")
    plt.xlabel("t")
    plt.ylabel(r"$E_n(t)=|c_n(t)|^2$")
    plt.title("1D Dirichlet-cavity NLS (FD): selected mode energies")
    plt.legend()
    plt.show()

    # -----------------------------
    # Plot 2: mode-index heatmap
    # -----------------------------
    plt.figure()
    extent = [t_eval[0], t_eval[-1], modes[0], modes[-1]]
    plt.imshow(
        En_log,
        aspect="auto",
        origin="lower",
        extent=extent,
        interpolation="nearest",
    )
    plt.xlabel("t")
    plt.ylabel("Mode index n")
    plt.title(r"Mode-index heatmap: $\log_{10}(E_n(t))$")
    plt.colorbar(label=r"$\log_{10}(E_n)$")
    plt.show()

    # -----------------------------
    # Plot 3: final intensity (full field)
    # -----------------------------
    psi_final_full = np.zeros(N, dtype=np.complex128)
    psi_final_full[1:-1] = Psi[:, -1]
    plt.figure()
    plt.plot(x, np.abs(psi_final_full)**2)
    plt.xlabel("x")
    plt.ylabel(r"$|\psi|^2$")
    plt.title("Final intensity")
    plt.show()

if __name__ == "__main__":
    run_fd_no_time_loop_with_heatmap()


RK4:   0%|          | 0/2120877 [00:00<?, ?step/s]

/var/folders/yf/_n54ygxn3c94qnd5krdc2pwr0000gn/T/ipykernel_80622/1726746328.py:54: RuntimeWarning: overflow encountered in square
  return Lin @ psi - 1j * g * (np.abs(psi)**2) * psi
/var/folders/yf/_n54ygxn3c94qnd5krdc2pwr0000gn/T/ipykernel_80622/1726746328.py:54: RuntimeWarning: invalid value encountered in multiply
  return Lin @ psi - 1j * g * (np.abs(psi)**2) * psi
/var/folders/yf/_n54ygxn3c94qnd5krdc2pwr0000gn/T/ipykernel_80622/1726746328.py:54: RuntimeWarning: overflow encountered in multiply
  return Lin @ psi - 1j * g * (np.abs(psi)**2) * psi
/var/folders/yf/_n54ygxn3c94qnd5krdc2pwr0000gn/T/ipykernel_80622/1726746328.py:86: RuntimeWarning: invalid value encountered in multiply
  k4 = rhs(t + dt, psi + dt * k3)
/var/folders/yf/_n54ygxn3c94qnd5krdc2pwr0000gn/T/ipykernel_80622/1726746328.py:88: RuntimeWarning: overflow encountered in multiply
  psi = psi + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
/var/folders/yf/_n54ygxn3c94qnd5krdc2pwr0000gn/T/ipykernel_80622/1726746328.py:8

KeyboardInterrupt: 